# Lesson 9: Model Evaluation

## Train/Test Split, Cross-Validation, Regression Metrics

In this notebook we will:
- Split data into train and test sets
- Implement k-fold cross-validation
- Compute MAE, MSE, RMSE, and R²
- Diagnose overfitting

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('ggplot')
%matplotlib inline

## 1. Basic Train/Test Split

In [ ]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Set Performance:")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE:  {mean_squared_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R²:   {r2_score(y_test, y_pred):.3f}")

## 2. Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(LinearRegression(), X, y, cv=cv, scoring='r2')

print(f"5-Fold CV R² scores: {cv_scores}")
print(f"Mean: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")

## 3. Prediction vs Actual Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title(f'Diabetes: Predictions vs Actual (R² = {r2_score(y_test, y_pred):.3f})')
plt.tight_layout()
plt.show()

## 4. Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred, residuals, alpha=0.6)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.show()

print(f"Mean residual: {np.mean(residuals):.4f}")
print(f"Std residual: {np.std(residuals):.4f}")

## 5. California Housing Full Example

In [ ]:
housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

model_h = LinearRegression()
model_h.fit(X_h_train, y_h_train)
y_h_pred = model_h.predict(X_h_test)

print("California Housing Performance:")
print(f"MAE:  ${mean_absolute_error(y_h_test, y_h_pred):.3f}k")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_h_test, y_h_pred)):.3f}k")
print(f"R²:   {r2_score(y_h_test, y_h_pred):.4f}")

cv_scores_h = cross_val_score(LinearRegression(), X_h, y_h, cv=10, scoring='r2')
print(f"\n10-Fold CV R²: {cv_scores_h.mean():.4f} (±{cv_scores_h.std():.4f})")

## Exercises

1. Compare CV R² of LinearRegression vs mean-predicting baseline on diabetes.
2. Write `evaluate_model(model, X, y, cv_folds=5)` returning MAE, RMSE, R².
3. Load California housing, train model, compute all metrics, create residual plots.